## LOAD DATASET

In [ ]:
import pandas as pd
import numpy as np
sales = pd.read_csv('sales.csv')

## FEATURE ENGINEERING

In [ ]:
PROMO_SCHEDULE = [
    ('spring_sale',   3, 18, 30, 12, True),
    ('mid_year',      6, 23, 29, 18, True),
    ('fall_launch',   8, 30, 32, 10, True),
    ('year_end',     11, 18, 45, 20, True),
    ('urban_blowout', 7, 30, 33, None, 'odd'),
    ('rural_special', 1, 30, 30, 15,   'odd'),
]

TET_DATES = {
    2013:'2013-02-10', 2014:'2014-01-31', 2015:'2015-02-19',
    2016:'2016-02-08', 2017:'2017-01-28', 2018:'2018-02-16',
    2019:'2019-02-05', 2020:'2020-01-25', 2021:'2021-02-12',
    2022:'2022-02-01', 2023:'2023-01-22', 2024:'2024-02-10',
}

VN_FIXED_HOLIDAYS = [
    (1,1,'new_year'), (3,8,'womens_day'), (4,30,'reunification'),
    (5,1,'labor_day'), (9,2,'national_day'), (10,20,'vn_womens_day'),
    (11,11,'dd_1111'), (12,12,'dd_1212'),
    (12,24,'christmas_eve'), (12,25,'christmas'),
]

In [ ]:
def build_features(dates):
    df = pd.DataFrame({'Date': dates})
    d = df['Date']

    # Calendar
    df['year'] = d.dt.year
    df['month'] = d.dt.month
    df['day'] = d.dt.day
    df['dow'] = d.dt.dayofweek
    df['doy'] = d.dt.dayofyear
    df['quarter'] = d.dt.quarter
    df['is_weekend'] = (df['dow'] >= 5).astype(int)
    df['days_to_eom'] = d.dt.days_in_month - df['day']
    df['days_from_som'] = df['day'] - 1
    df['dim'] = d.dt.days_in_month

    # Edge of month
    for k in [1, 2, 3]:
        df[f'is_last{k}'] = (df['days_to_eom'] <= k - 1).astype(int)
        df[f'is_first{k}'] = (df['days_from_som'] <= k - 1).astype(int)

    # Trend + regime
    df['t_days'] = (d - pd.Timestamp('2020-01-01')).dt.days
    df['t_years'] = df['t_days'] / 365.25
    df['regime_pre2019'] = (df['year'] <= 2018).astype(int)
    df['regime_2019'] = (df['year'] == 2019).astype(int)
    df['regime_post2019'] = (df['year'] >= 2020).astype(int)

    # Fourier
    TAU = 2 * np.pi
    for k in (1, 2, 3, 4, 5):
        df[f'sin_y{k}'] = np.sin(TAU * k * df['doy'] / 365.25)
        df[f'cos_y{k}'] = np.cos(TAU * k * df['doy'] / 365.25)
    for k in (1, 2):
        df[f'sin_w{k}'] = np.sin(TAU * k * df['dow'] / 7.0)
        df[f'cos_w{k}'] = np.cos(TAU * k * df['dow'] / 7.0)
    for k in (1, 2):
        df[f'sin_m{k}'] = np.sin(TAU * k * (df['day'] - 1) / df['dim'])
        df[f'cos_m{k}'] = np.cos(TAU * k * (df['day'] - 1) / df['dim'])

    # Holidays
    for (m, dd_, name) in VN_FIXED_HOLIDAYS:
        df[f'hol_{name}'] = ((df['month'] == m) & (df['day'] == dd_)).astype(int)

    # Tet distance
    tet_lut = {y: pd.Timestamp(v) for y, v in TET_DATES.items()}

    def nearest_tet_diff(dd):
        cands = [tet_lut.get(dd.year), tet_lut.get(dd.year - 1), tet_lut.get(dd.year + 1)]
        cands = [c for c in cands if c is not None]
        valid = [(dd - c).days for c in cands if abs((dd - c).days) <= 45]
        return min(valid) if valid else 999

    diffs = np.array([nearest_tet_diff(dd) for dd in d])
    df['tet_days_diff'] = diffs
    df['tet_in_7'] = (np.abs(diffs) <= 7).astype(int)
    df['tet_in_14'] = (np.abs(diffs) <= 14).astype(int)
    df['tet_before_7'] = ((diffs >= -7) & (diffs < 0)).astype(int)
    df['tet_after_7'] = ((diffs > 0) & (diffs <= 7)).astype(int)
    df['tet_on'] = (diffs == 0).astype(int)

    # Black Friday
    def is_bf(dd):
        if dd.month != 11: return 0
        last = pd.Timestamp(year=dd.year, month=11, day=30)
        last_fri = last - pd.Timedelta(days=(last.dayofweek - 4) % 7)
        return int(dd == last_fri)

    df['hol_black_friday'] = [is_bf(dd) for dd in d]

    # Promo windows
    yrs = sorted(set(df['year'].tolist()))
    for (name, sm, sd, dur, disc, recur) in PROMO_SCHEDULE:
        in_prom = np.zeros(len(df), dtype=int)
        since = np.full(len(df), -1.0)
        until = np.full(len(df), -1.0)
        discount = np.zeros(len(df))
        for y in range(min(yrs) - 1, max(yrs) + 2):
            if recur == 'odd' and y % 2 == 0: continue
            start = pd.Timestamp(year=y, month=sm, day=sd)
            end = start + pd.Timedelta(days=dur)
            mask = (d >= start) & (d <= end)
            in_prom[mask] = 1
            since[mask] = (d[mask] - start).dt.days
            until[mask] = (end - d[mask]).dt.days
            discount[mask] = disc or 0
        df[f'promo_{name}'] = in_prom
        df[f'promo_{name}_since'] = since
        df[f'promo_{name}_until'] = until
        df[f'promo_{name}_disc'] = discount

    df['is_odd_year'] = (df['year'] % 2).astype(int)
    return df

In [ ]:
full_feat = build_features(pd.date_range('2012-07-04', '2024-07-01'))
train = full_feat.head(len(full_feat)-548)
test = full_feat.tail(548)
train['Revenue'] = sales['Revenue'].values
train['COGS'] = sales['COGS'].values
train.to_csv('DL/features/train.csv', index=False)
test.to_csv('DL/features/test.csv', index=False)

## RUN MODEL & SAVE SUBMISSION

TRAIN FROM SCRATCH (~1.5 HOURS ON T4 GPU)

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt
import lightning.pytorch as pl
from pytorch_forecasting import TimeSeriesDataSet, TemporalFusionTransformer
from pytorch_forecasting.data import GroupNormalizer
from pytorch_forecasting.metrics import RMSE
import gc

pl.seed_everything(42, workers=True)

# ==========================================
# 1. LOAD DATASET
# ==========================================
df_train = pd.read_csv("DL/features/train.csv")
df_test = pd.read_csv("DL/features/test.csv")

df_train['is_test'] = 0
df_test['is_test'] = 1

df_full = pd.concat([df_train, df_test], ignore_index=True)
df_full['Date'] = pd.to_datetime(df_full['Date'])
df_full = df_full.sort_values('Date').reset_index(drop=True)
df_full = df_full[df_full['Date'].dt.year >= 2014].copy() # Bỏ giai đoạn 2012-2013 vì giai đoạn này nhiễu mạnh

df_full['time_idx'] = (df_full['Date'] - df_full['Date'].min()).dt.days
df_full['group'] = 'store_01'

# Scale Revenue và COGS xuống để chuẩn hóa
df_full['Revenue'] = (df_full['Revenue'].fillna(0) / 1e6).astype(float)
df_full['COGS'] = (df_full['COGS'].fillna(0) / 1e6).astype(float)

leakage_cols = ['Date', 'Revenue', 'COGS', 'time_idx', 'group', 'is_test']
known_reals = [c for c in df_full.columns
               if c not in leakage_cols
               and df_full[c].dtype in [np.float64, np.int64]]

df_full[known_reals] = df_full[known_reals].fillna(0)

# ==========================================
# 2. CẤU HÌNH & HÀM HUẤN LUYỆN
# ==========================================
ENCODER_LENGTH = 90 # Chỉ cho encoder nhìn 90 ngày (1 quý) trước để tránh tràn RAM
PREDICTION_LENGTH = len(df_test) # 548 ngày

def train_plot_save(target_col, max_epochs=30):
    train_data = df_full[df_full['is_test'] == 0].copy()

    training = TimeSeriesDataSet(
        train_data,
        time_idx="time_idx",
        target=target_col,
        group_ids=["group"],
        min_encoder_length=ENCODER_LENGTH,
        max_encoder_length=ENCODER_LENGTH,
        min_prediction_length=1,
        max_prediction_length=PREDICTION_LENGTH,
        static_categoricals=["group"],
        time_varying_known_reals=known_reals,
        time_varying_unknown_reals=[target_col],
        target_normalizer=GroupNormalizer(groups=["group"], transformation="softplus"),
        add_relative_time_idx=True,
        add_target_scales=True,
        add_encoder_length=True,
    )

    train_dataloader = training.to_dataloader(train=True, batch_size=64, num_workers=0)

    tft = TemporalFusionTransformer.from_dataset(
        training,
        learning_rate=0.005,
        hidden_size=16,
        hidden_continuous_size=8,
        attention_head_size=2,
        dropout=0.3,
        loss=RMSE(),
        optimizer="Adam"
    )

    trainer = pl.Trainer(
        max_epochs=max_epochs,
        accelerator="gpu",
        devices=1,
        precision="32-true",          # Dùng 32-bit để chống Overflow
        logger=False,
        enable_checkpointing=False,
        deterministic="warn"
    )

    # 1. Huấn luyện
    trainer.fit(tft, train_dataloaders=train_dataloader)
    trainer.save_checkpoint(f"DL/checkpoints/model_{target_col}.ckpt")

    # 2. Chuẩn bị dữ liệu Test
    encoder_data = train_data[train_data.time_idx >= train_data.time_idx.max() - ENCODER_LENGTH + 1]
    test_data = df_full[df_full['is_test'] == 1].copy()
    predict_data_raw = pd.concat([encoder_data, test_data], ignore_index=True)

    predict_dataset = TimeSeriesDataSet.from_dataset(
        training, predict_data_raw, predict=True, stop_randomization=True
    )
    predict_dataloader = predict_dataset.to_dataloader(train=False, batch_size=1, num_workers=0)

    # 3. Lấy dự báo số liệu để nộp bài
    preds = tft.predict(predict_dataloader, mode="prediction", return_x=False).numpy().flatten()

    # 4. Tạo và lưu biểu đồ forecast
    predict_output = tft.predict(predict_dataloader, mode="raw", return_x=True)
    raw_predictions = predict_output[0]
    x = predict_output[1]

    fig = tft.plot_prediction(x, raw_predictions, idx=0, add_loss_to_title=False)

    plt.title(f"TFT Forecast vs Actual: {target_col.upper()}", fontsize=14, fontweight='bold')
    plt.xlabel("Time Index (Ngày)", fontsize=12)
    plt.ylabel(f"Giá trị ({target_col})", fontsize=12)
    plt.tight_layout()

    fig.savefig(f"plots/DL/forecast_{target_col}.png", dpi=300, bbox_inches='tight')
    plt.close(fig)

    # 5. Giải thích (interpret) mô hình
    interpretation = tft.interpret_output(raw_predictions, reduction="sum")
    figs = tft.plot_interpretation(interpretation)

    for plot_name, fig in figs.items():
        fig.suptitle(f"TFT Interpretation: {plot_name.replace('_', ' ').title()} ({target_col})", fontsize=12)
        plt.tight_layout()
        fig.savefig(f"plots/DL/{plot_name}_{target_col}.png", dpi=300, bbox_inches='tight')
        plt.close(fig)

    # 6. Dọn dẹp RAM
    del tft, trainer, train_dataloader, training, predict_dataset, predict_dataloader, raw_predictions, x
    gc.collect()
    torch.cuda.empty_cache()

    return preds

# ==========================================
# 3. THỰC THI & XUẤT KẾT QUẢ
# ==========================================
pred_revenue = train_plot_save('Revenue', max_epochs=30)
pred_cogs = train_plot_save('COGS', max_epochs=30)

sub_mask = df_full['is_test'] == 1
submission = pd.DataFrame({
    'Date': df_full.loc[sub_mask, 'Date'].dt.strftime('%Y-%m-%d'),
    'Revenue': pred_revenue * 1e6, # Scale lên thang chuẩn
    'COGS': pred_cogs * 1e6
})

submission.to_csv('submissions/submission_dl.csv', index=False)
print("\nFile submission và biểu đồ đã hoàn thành!")

REPRODUCE USING CHECKPOINTS

In [ ]:
import os
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import torch
import gc

from pytorch_forecasting import TimeSeriesDataSet, TemporalFusionTransformer
from pytorch_forecasting.data import GroupNormalizer

# ==========================================
# 1. LOAD DATASET
# ==========================================
df_train = pd.read_csv("DL/features/train.csv")
df_test = pd.read_csv("DL/features/test.csv")

df_train['is_test'] = 0
df_test['is_test'] = 1

df_full = pd.concat([df_train, df_test], ignore_index=True)
df_full['Date'] = pd.to_datetime(df_full['Date'])
df_full = df_full.sort_values('Date').reset_index(drop=True)
df_full = df_full[df_full['Date'].dt.year >= 2014].copy()

df_full['time_idx'] = (df_full['Date'] - df_full['Date'].min()).dt.days
df_full['group'] = 'store_01'

df_full['Revenue'] = (df_full['Revenue'].fillna(0) / 1e6).astype(float)
df_full['COGS'] = (df_full['COGS'].fillna(0) / 1e6).astype(float)

leakage_cols = ['Date', 'Revenue', 'COGS', 'time_idx', 'group', 'is_test']
known_reals = [c for c in df_full.columns
               if c not in leakage_cols
               and df_full[c].dtype in [np.float64, np.int64]]

df_full[known_reals] = df_full[known_reals].fillna(0)

# Thông số cố định (Bắt buộc phải khớp với lúc Train)
ENCODER_LENGTH = 90
PREDICTION_LENGTH = len(df_test)

# ==========================================
# 2. HÀM DỰ BÁO TỪ CHECKPOINT
# ==========================================
def predict_from_checkpoint(target_col, checkpoint_path):
    print(f"\n[{target_col.upper()}] Đang xử lý từ Checkpoint: {checkpoint_path}")

    # 1. Tái tạo lại cấu trúc tập Train (KHÔNG train lại, chỉ lấy Metadata/Normalizer)
    train_data = df_full[df_full['is_test'] == 0].copy()

    training = TimeSeriesDataSet(
        train_data,
        time_idx="time_idx",
        target=target_col,
        group_ids=["group"],
        min_encoder_length=ENCODER_LENGTH,
        max_encoder_length=ENCODER_LENGTH,
        min_prediction_length=1,
        max_prediction_length=PREDICTION_LENGTH,
        static_categoricals=["group"],
        time_varying_known_reals=known_reals,
        time_varying_unknown_reals=[target_col],
        target_normalizer=GroupNormalizer(groups=["group"], transformation="softplus"),
        add_relative_time_idx=True,
        add_target_scales=True,
        add_encoder_length=True,
    )

    # 2. Lấy dữ liệu 90 ngày cuối của Train + toàn bộ Test để làm input cho dự báo
    encoder_data = train_data[train_data.time_idx >= train_data.time_idx.max() - ENCODER_LENGTH + 1]
    test_data = df_full[df_full['is_test'] == 1].copy()
    predict_data_raw = pd.concat([encoder_data, test_data], ignore_index=True)

    predict_dataset = TimeSeriesDataSet.from_dataset(
        training, predict_data_raw, predict=True, stop_randomization=True
    )
    # batch_size=1 vì ta chỉ dự báo cho 1 group duy nhất (store_01)
    predict_dataloader = predict_dataset.to_dataloader(train=False, batch_size=1, num_workers=0)

    # 3. Load checkpoint và dự báo
    print(f"[{target_col.upper()}] Đang load checkpoint và dự báo...")

    # Load model từ file (Mặc định nó sẽ tự động nhận diện GPU nếu có)
    tft = TemporalFusionTransformer.load_from_checkpoint(checkpoint_path)

    # Đóng băng trọng số, chuyển sang chế độ đánh giá (tắt Dropout)
    tft.eval()

    # Thực thi dự báo
    preds = tft.predict(predict_dataloader, mode="prediction", return_x=False).numpy().flatten()

    # 4. Tạo và lưu biểu đồ forecast
    predict_output = tft.predict(predict_dataloader, mode="raw", return_x=True)
    raw_predictions = predict_output[0]
    x = predict_output[1]

    fig = tft.plot_prediction(x, raw_predictions, idx=0, add_loss_to_title=False)

    plt.title(f"TFT Forecast vs Actual: {target_col.upper()}", fontsize=14, fontweight='bold')
    plt.xlabel("Time Index (Ngày)", fontsize=12)
    plt.ylabel(f"Giá trị ({target_col})", fontsize=12)
    plt.tight_layout()

    fig.savefig(f"plots/DL/forecast_{target_col}.png", dpi=300, bbox_inches='tight')
    plt.close(fig)

    # 5. Giải thích (interpret) mô hình
    interpretation = tft.interpret_output(raw_predictions, reduction="sum")
    figs = tft.plot_interpretation(interpretation)

    for plot_name, fig in figs.items():
        fig.suptitle(f"TFT Interpretation: {plot_name.replace('_', ' ').title()} ({target_col})", fontsize=12)
        plt.tight_layout()
        fig.savefig(f"plots/DL/{plot_name}_{target_col}.png", dpi=300, bbox_inches='tight')
        plt.close(fig)

    # 6. Dọn dẹp RAM
    del tft, training, predict_dataset, predict_dataloader
    gc.collect()
    torch.cuda.empty_cache()

    print(f"[{target_col.upper()}] Trích xuất dự báo thành công!")
    return preds

# ==========================================
# 3. THỰC THI & XUẤT KẾT QUẢ
# ==========================================
ckpt_revenue = "DL/checkpoints/model_Revenue.ckpt"
ckpt_cogs = "DL/checkpoints/model_COGS.ckpt"

# Gọi hàm dự báo
pred_revenue = predict_from_checkpoint('Revenue', ckpt_revenue)
pred_cogs = predict_from_checkpoint('COGS', ckpt_cogs)

# Tạo file submission
sub_mask = df_full['is_test'] == 1
submission = pd.DataFrame({
    'Date': df_full.loc[sub_mask, 'Date'].dt.strftime('%Y-%m-%d'),
    'Revenue': pred_revenue * 1e6,
    'COGS': pred_cogs * 1e6
})

submission_path = 'submissions/submission_dl.csv'
submission.to_csv(submission_path, index=False)
print("\nFile submission và biểu đồ đã hoàn thành!")